In [ ]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 60.4 MB/s eta 0:00:00


# Faster R-CNN

In [ ]:
import torch
import yaml
import os
import random
import shutil
import math
from huggingface_hub import hf_hub_download
import xml.etree.ElementTree as ET
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as F
from torchmetrics.detection import MeanAveragePrecision
from torchvision.models.detection import (FasterRCNN_ResNet50_FPN_Weights,
                                          fasterrcnn_resnet50_fpn)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.utils import draw_bounding_boxes
from torchvision.io import ImageReadMode, decode_jpeg, read_file
from PIL import Image
import time
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Setup đường dẫn và thiết bị

In [ ]:
REPO_ID = "bsmock/pubtables-1m"
IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"
MODEL = "/content/best_faster_rcnn.pt"
ROOT = "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
IMGSZ = 640
BATCH_SIZE = 32
ACCUM = 4
LR = 0.02
EPOCHS = 10
WARMUP_STEPS = 3
EVAL_EVERY = 1
FAST_HEADS = True

MAX_TRAIN, MAX_VAL, MAX_TEST = None, None, None
NUM_WORKERS = min(os.cpu_count() or 2, 8)

random.seed(SEED)
torch.manual_seed(SEED)
print("Device:", DEVICE, "| CPU:", os.cpu_count(), "| workers:", NUM_WORKERS)

Device: cuda | CPU: 12 | workers: 8


Tải tập hình ảnh huấn luyện của dataset, không chạy cell này nếu chỉ muốn test

In [ ]:
os.makedirs("cache", exist_ok = True)
path = hf_hub_download(repo_id=REPO_ID, filename="PubTables-1M-Detection_Images_Train_Part1.tar.gz", local_dir="cache", repo_type="dataset")
dest = os.path.join(IMAGES_DIR, "train")
os.makedirs(dest, exist_ok = True)
!tar -xf {path} -C {dest}
!rm {path}
path = hf_hub_download(repo_id=REPO_ID, filename="PubTables-1M-Detection_Images_Train_Part2.tar.gz", local_dir="cache", repo_type="dataset")
!tar -xf {path} -C {dest}
!rm {path}
dest = os.path.join(IMAGES_DIR, "val")
os.makedirs(dest, exist_ok = True)
path = hf_hub_download(repo_id=REPO_ID, filename="PubTables-1M-Detection_Images_Val.tar.gz", local_dir="cache", repo_type="dataset")
!tar -xf {path} -C {dest}
!rm {path}
!rm -rf "cache"

PubTables-1M-Detection_Images_Train_Part(…): reconstructing file:   0%|          |  0.00B / 30.8GB            

PubTables-1M-Detection_Images_Train_Part(…): downloading bytes:           |  0.00B            

PubTables-1M-Detection_Images_Train_Part(…): reconstructing file:   0%|          |  0.00B / 29.3GB            

PubTables-1M-Detection_Images_Train_Part(…): downloading bytes:           |  0.00B            

PubTables-1M-Detection_Images_Val.tar.gz: reconstructing file:   0%|          |  0.00B / 7.52GB            

PubTables-1M-Detection_Images_Val.tar.gz: downloading bytes:           |  0.00B            

Tải tập hình ảnh kiểm thử của dataset

In [ ]:
os.makedirs("cache", exist_ok = True)
dest = os.path.join(IMAGES_DIR, "test")
os.makedirs(dest, exist_ok = True)
path = hf_hub_download(repo_id=REPO_ID, filename="PubTables-1M-Detection_Images_Test.tar.gz", local_dir="cache", repo_type="dataset")
!tar -xf {path} -C {dest}
!rm {path}
!rm -rf "cache"

PubTables-1M-Detection_Images_Test.tar.g(…): reconstructing file:   0%|          |  0.00B / 7.44GB            

PubTables-1M-Detection_Images_Test.tar.g(…): downloading bytes:           |  0.00B            

Tải Annotation

In [ ]:
os.makedirs("cache", exist_ok = True)
dest = os.path.join(LABELS_DIR, "train")
os.makedirs(dest, exist_ok = True)
path = hf_hub_download(repo_id=REPO_ID, filename="PubTables-1M-Detection_Annotations_Train.tar.gz", local_dir="cache", repo_type="dataset")
!tar -xf {path} -C {dest}
!rm {path}
dest = os.path.join(LABELS_DIR, "val")
os.makedirs(dest, exist_ok = True)
path = hf_hub_download(repo_id=REPO_ID, filename="PubTables-1M-Detection_Annotations_Val.tar.gz", local_dir="cache", repo_type="dataset")
!tar -xf {path} -C {dest}
!rm {path}
dest = os.path.join(LABELS_DIR, "test")
os.makedirs(dest, exist_ok = True)
path = hf_hub_download(repo_id=REPO_ID, filename="PubTables-1M-Detection_Annotations_Test.tar.gz", local_dir="cache", repo_type="dataset")
!tar -xf {path} -C {dest}
!rm {path}
!rm -rf "cache"

PubTables-1M-Detection_Annotations_Train(…): reconstructing file:   0%|          |  0.00B / 40.1MB            

PubTables-1M-Detection_Annotations_Train(…): downloading bytes:           |  0.00B            

PubTables-1M-Detection_Annotations_Val.t(…): reconstructing file:   0%|          |  0.00B / 5.02MB            

PubTables-1M-Detection_Annotations_Val.t(…): downloading bytes:           |  0.00B            

PubTables-1M-Detection_Annotations_Test.(…): reconstructing file:   0%|          |  0.00B / 4.49MB            

PubTables-1M-Detection_Annotations_Test.(…): downloading bytes:           |  0.00B            

In [ ]:
!rm -rf "data/images/train_subset"
!rm -rf "data/images/val_subset"
!rm -rf "data/images/test_subset"
!rm -rf "data/labels/train_subset"
!rm -rf "data/labels/val_subset"
!rm -rf "data/labels/test_subset"

In [ ]:
base_dir = "/content/data"
subset_sizes = {
    "train": int(len(os.listdir(os.path.join(base_dir, "images", "train"))) * 0.05),
    "val": int(len(os.listdir(os.path.join(base_dir, "images", "val"))) * 0.05),
    "test": int(len(os.listdir(os.path.join(base_dir, "images", "test"))) * 0.05),
}
random.seed(42)
for split, size in subset_sizes.items():
    src_img_dir = os.path.join(base_dir, "images", split)
    src_lbl_dir = os.path.join(base_dir, "labels", split)

    dst_img_dir = os.path.join(base_dir, "images", f"{split}_subset")
    dst_lbl_dir = os.path.join(base_dir, "labels", f"{split}_subset")

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    for folder in [dst_img_dir, dst_lbl_dir]:
        if os.path.exists(folder):
            shutil.rmtree(folder)
        os.makedirs(folder, exist_ok=True)

    all_images = [f for f in os.listdir(src_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    sampled_images = random.sample(all_images, min(size, len(all_images)))

    copied_count = 0
    for img_name in sampled_images:
        shutil.copy(os.path.join(src_img_dir, img_name), os.path.join(dst_img_dir, img_name))

        label_name = os.path.splitext(img_name)[0] + ".xml"
        label_path = os.path.join(src_lbl_dir, label_name)
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(dst_lbl_dir, label_name))

        copied_count += 1

    print(f"Created '{split}_subset' with {copied_count} images and corresponding labels.")

Created 'train_subset' with 23029 images and corresponding labels.
Created 'val_subset' with 2879 images and corresponding labels.
Created 'test_subset' with 2856 images and corresponding labels.


## Tạo class Dataset

In [ ]:
CLASSES = {"table": 1, "table rotated": 2}


class PubTablesSet(Dataset):
    def __init__(self, split, max_size=None):
        self.img_dir = f"{ROOT}/images/{split}"
        ann_dir = f"{ROOT}/labels/{split}"
        self.names, self.targets = [], []
        for f in sorted(os.listdir(self.img_dir)):      # cắt sớm, không kiểm tra toàn bộ thư mục
            if not f.endswith(".jpg"):
                continue
            xml = f"{ann_dir}/{f[:-4]}.xml"
            if not os.path.exists(xml):
                continue
            self.names.append(f[:-4])
            self.targets.append(self._parse(xml))
            if max_size and len(self.names) >= max_size:
                break

    @staticmethod
    def _parse(xml_path):
        boxes, labels = [], []
        for obj in ET.parse(xml_path).getroot().iter("object"):
            cls = obj.find("name").text
            if cls not in CLASSES:
                continue
            b = obj.find("bndbox")
            x1, y1, x2, y2 = (float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
            if x2 - x1 > 1 and y2 - y1 > 1:
                boxes.append([x1, y1, x2, y2])
                labels.append(CLASSES[cls])              # 0 = background
        return {"boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                "labels": torch.tensor(labels, dtype=torch.int64)}

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        img = decode_jpeg(read_file(f"{self.img_dir}/{self.names[i]}.jpg"), mode=ImageReadMode.RGB)  # uint8 CHW
        return img, self.targets[i]


def collate(batch):
    return tuple(zip(*batch))


def to_dev(imgs, targets=None):
    """uint8 CPU -> float GPU [0,1]."""
    imgs = [i.to(DEVICE, non_blocking=True).float().div_(255) for i in imgs]
    if targets is None:
        return imgs
    return imgs, [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]


def make_loader(ds, batch_size, shuffle=False, persistent=True):
    kw = dict(batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, collate_fn=collate,
              pin_memory=DEVICE.type == "cuda")
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=persistent, prefetch_factor=4)
    if shuffle:
        kw["generator"] = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, **kw)


train_ds = PubTablesSet("train_subset", MAX_TRAIN)
val_ds = PubTablesSet("val_subset", MAX_VAL)
test_ds = PubTablesSet("test_subset", MAX_TEST)

train_dl = make_loader(train_ds, BATCH_SIZE, shuffle=True)
val_dl = make_loader(val_ds, BATCH_SIZE)
test_dl = make_loader(test_ds, BATCH_SIZE)
print(f"train/val/test: {len(train_ds)}/{len(val_ds)}/{len(test_ds)} ảnh | "
      f"{len(train_dl)} micro-batch/epoch = {math.ceil(len(train_dl) / ACCUM)} bước optimizer/epoch")

train/val/test: 23029/2879/2856 ảnh | 720 micro-batch/epoch = 180 bước optimizer/epoch


## Setup model

In [ ]:
extra = {}
if FAST_HEADS:   # bảng chỉ có vài vật thể/ảnh nên không cần 1000-2000 proposal
    extra = dict(rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=500,
                 rpn_pre_nms_top_n_test=500, rpn_post_nms_top_n_test=100,
                 box_batch_size_per_image=128)

model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,      # COCO pretrained, giống YOLO
    min_size=IMGSZ, max_size=IMGSZ,                       # cạnh dài = 640 như imgsz của YOLO
    box_score_thresh=0.001, box_detections_per_img=100,   # cùng ngưỡng đánh giá với YOLO (conf=0.001, max_det=100)
    **extra)
in_feat = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, len(CLASSES) + 1)
model.to(DEVICE)
model.to(memory_format=torch.channels_last)

params = [p for p in model.parameters() if p.requires_grad]
optim = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=5e-4)

steps_per_epoch = math.ceil(len(train_dl) / ACCUM)
total_steps = EPOCHS * steps_per_epoch


def lr_lambda(s):   # warmup tuyến tính + cosine, sàn 1% (tương tự lrf của YOLO)
    if s < WARMUP_STEPS:
        return (s + 1) / WARMUP_STEPS
    p = (s - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return max(0.01, 0.5 * (1 + math.cos(math.pi * min(p, 1.0))))


sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox")
    for imgs, targets in loader:
        with torch.autocast(DEVICE.type, enabled=use_amp):
            outs = model(to_dev(imgs))
        preds = [{"boxes": o["boxes"].float().cpu(), "scores": o["scores"].float().cpu(),
                  "labels": o["labels"].cpu()} for o in outs]
        metric.update(preds, list(targets))
    return {k: float(v) for k, v in metric.compute().items() if v.numel() == 1}


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def benchmark_speed(dataset, batch_size, max_images=200, warmup=3):
    """Chỉ đo thời gian model (đã gồm transform + NMS bên trong), dùng FP16 như lúc train."""
    model.eval()
    dl = make_loader(dataset, batch_size, persistent=False)
    n_img, t_total = 0, 0.0
    for i, (imgs, _) in enumerate(dl):
        imgs = to_dev(imgs)
        sync()
        t0 = time.perf_counter()
        with torch.autocast(DEVICE.type, enabled=use_amp):
            model(imgs)
        sync()
        dt = time.perf_counter() - t0
        if i >= warmup:                      # bỏ qua vài batch đầu (GPU khởi động)
            t_total += dt
            n_img += len(imgs)
        if n_img >= max_images:
            break
    return t_total / n_img * 1000, n_img / t_total

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 219MB/s]


## Train model

In [ ]:
best = -1.0
epoch_times, val_times = [], []
train_start = time.perf_counter()
n_mb = len(train_dl)

for epoch in range(1, EPOCHS + 1):
    sync()
    epoch_start = time.perf_counter()
    model.train()
    total = torch.zeros((), device=DEVICE)          # cộng dồn trên GPU, tránh loss.item() mỗi iter
    optim.zero_grad(set_to_none=True)
    for i, (imgs, targets) in enumerate(train_dl, 1):
        imgs, targets = to_dev(imgs, targets)
        group = min(ACCUM, n_mb - ((i - 1) // ACCUM) * ACCUM)   # số micro-batch của nhóm hiện tại
        with torch.autocast(DEVICE.type, enabled=use_amp):
            loss = sum(model(imgs, targets).values())
        scaler.scale(loss / group).backward()
        total += loss.detach()
        if i % ACCUM == 0 or i == n_mb:
            scaler.step(optim)
            scaler.update()
            optim.zero_grad(set_to_none=True)
            sched.step()
    sync()
    epoch_times.append(time.perf_counter() - epoch_start)

    msg = f"Epoch {epoch}/{EPOCHS} | loss {total.item() / n_mb:.4f} | train {epoch_times[-1]:.1f}s"
    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
        t0 = time.perf_counter()
        m = evaluate(val_dl)
        val_times.append(time.perf_counter() - t0)
        msg += (f" | val mAP@0.5 {m['map_50']:.4f} | mAP@[.5:.95] {m['map']:.4f} | val {val_times[-1]:.1f}s")
        print(msg)
        if m["map_50"] > best:
            best = m["map_50"]
            torch.save(model.state_dict(), MODEL)
            print(f"  -> lưu model tốt nhất: {MODEL}")
    else:
        print(msg)

sync()
total_wall = time.perf_counter() - train_start      # đo ngay khi train xong (không tính thời gian chờ giữa các cell)

Epoch 1/10 | loss 0.1689 | train 807.1s | val mAP@0.5 0.8431 | mAP@[.5:.95] 0.5937 | val 57.5s
  -> lưu model tốt nhất: /content/best_faster_rcnn.pt
Epoch 2/10 | loss 0.0653 | train 804.0s | val mAP@0.5 0.8913 | mAP@[.5:.95] 0.6296 | val 56.4s
  -> lưu model tốt nhất: /content/best_faster_rcnn.pt
Epoch 3/10 | loss 0.0535 | train 802.9s | val mAP@0.5 0.9754 | mAP@[.5:.95] 0.7600 | val 56.3s
  -> lưu model tốt nhất: /content/best_faster_rcnn.pt
Epoch 4/10 | loss 0.0417 | train 802.6s | val mAP@0.5 0.9949 | mAP@[.5:.95] 0.8490 | val 56.4s
  -> lưu model tốt nhất: /content/best_faster_rcnn.pt
Epoch 5/10 | loss 0.0356 | train 801.7s | val mAP@0.5 0.9950 | mAP@[.5:.95] 0.8835 | val 56.3s
  -> lưu model tốt nhất: /content/best_faster_rcnn.pt
Epoch 6/10 | loss 0.0314 | train 804.0s | val mAP@0.5 0.9949 | mAP@[.5:.95] 0.8925 | val 56.2s
Epoch 7/10 | loss 0.0285 | train 803.7s | val mAP@0.5 0.9950 | mAP@[.5:.95] 0.9184 | val 56.3s
  -> lưu model tốt nhất: /content/best_faster_rcnn.pt
Epoch 8/10 

## TEST

In [ ]:
model.load_state_dict(torch.load(MODEL, map_location=DEVICE))
m = evaluate(test_dl)
ms1, fps1 = benchmark_speed(test_ds, batch_size=1, max_images=100)
ms16, fps16 = benchmark_speed(test_ds, batch_size=16, max_images=200)

print("\n=============== BENCHMARK ===============")
print(f"Thiết bị            : {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}")
print(f"Số ảnh train / test : {len(train_dl.dataset)} / {len(test_dl.dataset)}")
print(f"imgsz / batch hiệu dụng : {IMGSZ} / {BATCH_SIZE * ACCUM}  (FAST_HEADS={FAST_HEADS})")
print("--- Thời gian huấn luyện ---")
print(f"Trung bình / epoch  : {sum(epoch_times) / len(epoch_times):.1f} s (chỉ train)")
print(f"Tổng train {EPOCHS} epoch  : {sum(epoch_times) / 60:.1f} phút (chưa tính val)")
print(f"Tổng val            : {sum(val_times) / 60:.1f} phút")
print(f"Tổng cả quá trình   : {total_wall / 60:.1f} phút (gồm val + lưu model)")
print("--- Tốc độ suy luận (test, FP16, gồm transform + NMS) ---")
print(f"Batch=1  (độ trễ)    : {ms1:.1f} ms/ảnh | {fps1:.1f} FPS")
print(f"Batch=16 (thông lượng): {ms16:.1f} ms/ảnh | {fps16:.1f} FPS")
print("--- Độ chính xác trên TEST ---")
print(f"mAP@[.50:.95]       : {m['map']:.4f}")
print(f"mAP@0.50            : {m['map_50']:.4f}")
print(f"mAP@0.75            : {m['map_75']:.4f}")
print(f"mAR@100             : {m['mar_100']:.4f}")
print("==========================================")


=============== BENCHMARK ===============
Thiết bị            : NVIDIA L4
Số ảnh train / test : 23029 / 2856
imgsz / batch hiệu dụng : 640 / 128  (FAST_HEADS=True)
--- Thời gian huấn luyện ---
Trung bình / epoch  : 803.9 s (chỉ train)
Tổng train 10 epoch  : 134.0 phút (chưa tính val)
Tổng val            : 9.4 phút
Tổng cả quá trình   : 143.4 phút (gồm val + lưu model)
--- Tốc độ suy luận (test, FP16, gồm transform + NMS) ---
Batch=1  (độ trễ)    : 27.7 ms/ảnh | 36.0 FPS
Batch=16 (thông lượng): 17.6 ms/ảnh | 56.7 FPS
--- Độ chính xác trên TEST ---
mAP@[.50:.95]       : 0.8943
mAP@0.50            : 0.9876
mAP@0.75            : 0.9876
mAR@100             : 0.9235


### Tải xuống file model tốt nhất và tự động tắt Runtime

In [ ]:
import time
import os
from google.colab import files
from google.colab import runtime

# Đường dẫn file weight tốt nhất từ thư mục chạy của Faster R-CNN
best_weight_src = MODEL

if os.path.exists(best_weight_src):
    print(f"Đang tải file {best_weight_src} về máy tính của bạn...")
    files.download(best_weight_src)
    # Chờ một lúc để tiến trình tải hoàn tất trước khi ngắt runtime
else:
    print("Không tìm thấy file weight để tải về. Vui lòng kiểm tra lại quá trình train.")

Đang tải file /content/best_faster_rcnn.pt về máy tính của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đang chờ 5 phút...
Đang tiến hành tắt và xóa runtime...


In [ ]:
print("Đang chờ 5 phút...")
time.sleep(300)
print("Đang tiến hành tắt và xóa runtime...")
runtime.unassign()